In [ ]:
import torch
import matplotlib.pyplot as plt
import EIANN.utils as ut
import EIANN.plot as pt
from EIANN._network import build_EIANN_from_config
import numpy as np
%load_ext autoreload
%autoreload 2

pt.update_plot_defaults()

train_dataloader, train_dataloader_CL1_full, train_dataloader_CL2_full, train_dataloader_CL1, train_dataloader_CL2, train_sub_dataloader, val_dataloader, test_dataloader, data_generator = ut.get_MNIST_dataloaders(sub_dataloader_size=10_000, classes=[0,1,2,3,4])

epochs = 1
data_seed = 0
network_seed = 42

## FF Backprop network

### Train phase 1

In [ ]:
# Build network
config_path = "../config/MNIST_templates/EIANN_1_hidden_mnist_almost_backprop.yaml"
network = build_EIANN_from_config(config_path, network_seed=66049)

# data_generator.manual_seed(257)
# network.train(train_dataloader_CL1, 
#                 test_dataloader, 
#                 epochs=1,
#                 val_interval=(0,-1,100),
#                 store_history=True, 
#                 store_params=False,
#                 status_bar=True)
# network.save("saved_networks/EIANN_1_hidden_mnist_VanBackprop_66049_257_20000stepsCL_softplus.pkl")
network.load("saved_networks/EIANN_1_hidden_mnist_VanBackprop_66049_257_10000stepsCL_softplus.pkl")

# pt.plot_batch_accuracy(network, test_dataloader)

plt.plot(network.val_loss_history)
pt.plot_batch_accuracy(network, test_dataloader, population=network.H1.E)
pt.plot_hidden_weights(network.module_dict['H1E_InputE'].weight, sort=True)

### Train phase 2

In [ ]:
network.phase1_params = {name:param.detach().clone() for name,param in network.named_parameters() if param.requires_grad}
network.diag_fisher = ut.compute_diag_fisher(network, train_dataloader_CL1_full)
network.ewc_lambda = 8000

from EIANN import rules
# Replace Backprop with Elastic Weight Consolidation in the backward methods
for i,method in enumerate(network.backward_methods):
    if str(method) == str(rules.Backprop.backward):
        network.backward_methods[i] = rules.Backprop_EWC.backward


In [ ]:
data_generator.manual_seed(257)
network.train(train_dataloader_CL2, 
                test_dataloader, 
                epochs=1,
                val_interval=(0,-1,100),
                store_history=True,
                store_params=False,
                status_bar=True)
network.save("saved_networks/EIANN_1_hidden_mnist_VanBackprop_66049_257_10000stepsCL2_softplus.pkl")
# network.load("saved_networks/EIANN_1_hidden_mnist_VanBackprop_66049_257_10000stepsCL2_softplus.pkl")

# pt.plot_batch_accuracy(network, test_dataloader)

plt.plot(network.val_loss_history)
pt.plot_batch_accuracy(network, test_dataloader, population=network.H1.E)
pt.plot_hidden_weights(network.module_dict['H1E_InputE'].weight, sort=True)

## Backprop Dale

In [ ]:
# Build network
config_path = "../config/MNIST_templates/EIANN_1_hidden_mnist_almost_backprop_EI.yaml"
network = build_EIANN_from_config(config_path, network_seed=66049)

# data_generator.manual_seed(257)
# network.train(train_dataloader_CL1, 
#                 test_dataloader, 
#                 epochs=1,
#                 val_interval=(0,-1,100),
#                 store_history=True, 
#                 store_params=False,
#                 status_bar=True)
# network.save("saved_networks/EIANN_1_hidden_mnist_bpDale_randomFBI_66049_257_10000stepsCL_relu.pkl")
network.load("saved_networks/EIANN_1_hidden_mnist_bpDale_randomFBI_66049_257_10000stepsCL_relu.pkl")

# pt.plot_batch_accuracy(network, test_dataloader)
plt.plot(network.val_loss_history)
pt.plot_batch_accuracy(network, test_dataloader, population=network.H1.E)
pt.plot_hidden_weights(network.module_dict['H1E_InputE'].weight, sort=True)

In [ ]:
network.phase1_params = {name:param.detach().clone() for name,param in network.named_parameters() if param.requires_grad}
network.diag_fisher = ut.compute_diag_fisher(network, train_dataloader_CL1_full)
network.ewc_lambda = 100000

from EIANN import rules
# Replace Backprop with Elastic Weight Consolidation in the backward methods
for i,method in enumerate(network.backward_methods):
    if str(method) == str(rules.Backprop.backward):
        network.backward_methods[i] = rules.Backprop_EWC.backward


In [ ]:
# data_generator.manual_seed(257)
# network.train(train_dataloader_CL2, 
#                 test_dataloader, 
#                 epochs=1,
#                 val_interval=(0,-1,100),
#                 store_history=True, 
#                 store_params=False,
#                 status_bar=True)
# network.save("saved_networks/EIANN_1_hidden_mnist_bpDale_randomFBI_66049_257_10000stepsCL2_relu.pkl")
network.load("saved_networks/EIANN_1_hidden_mnist_bpDale_randomFBI_66049_257_10000stepsCL2_relu.pkl")

# pt.plot_batch_accuracy(network, test_dataloader)

plt.plot(network.val_loss_history)
pt.plot_batch_accuracy(network, test_dataloader, population=network.H1.E)
pt.plot_hidden_weights(network.module_dict['H1E_InputE'].weight, sort=True)